In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

Environment Setup and State Schema

In [1]:
from typing import TypedDict, Optional, Dict, Any
import pandas as pd

#defining the state-schema
class DataScienceState(TypedDict):
    dataset_path: str
    dataset_summary: Optional[str]
    missing_values_summary: Optional[str]
    requires_imputation: bool
    imputation_strategy: Optional[str]
    scaling_strategy: Optional[str]
    hyperparameters: Optional[Dict[str,Any]]

print("DataScienceState schema defined sucessfully")

DataScienceState schema defined sucessfully


LLM & Tool initialization

In [7]:
from langchain_ollama import ChatOllama
from langchain_experimental.tools import PythonREPLTool
from langchain_core.prompts import PromptTemplate

#prototying with our local model
llm = ChatOllama(model = "qwen2.5-coder:3b", temperature =0)

#initialize the tool
python_repl = PythonREPLTool()

#Define a strict code-extraction helper function
def extract_code_block(llm_response: str) -> str:
    """Parses teh LLM output to extract only the executable python code."""
    if "```python" in llm_response:
        code = llm_response.split("```python")[1].split("```")[0].strip()
    elif "```" in llm_response:
        code = llm_response.split("```")[1].split("```")[0].strip()
    else:
        #fallback if the model forgets markdown formatiing
        code = llm_response.replace("Action: python_repl_ast", "").replace("Action Input:", "").strip()
    return code

print("LLM and REPL Tool initialized")

LLM and REPL Tool initialized


In [11]:
def eda_node(state: DataScienceState) -> DataScienceState:
    print("--Entering EDA Node--")
    dataset_path = state["dataset_path"]

    #1. Prompt engineering specifically for code generation without hallucinations 
    eda_prompt = PromptTemplate.from_template("""
        You are a senior data scientist. A dataset is located at '{filepath}'.
        Write a single block of Python code using pandas to do the following:
        1. Load the dataset.
        2. Print the dataframe info.
        3. Print the missing values sum for each column.
        4. Save a matplotlib distribution plot of the numerical columns to './plots/eda_distributions.png'.
        
        CRITICAL INSTRUCTION:
        Return ONLY valid Python code inside a ```python ``` markdown block. 
        Do not explain the code. Do not use 'Action Input'. Just write the code.
        """)

    #invoke LLM  and extract code
    chain = eda_prompt | llm
    response = chain.invoke({"filepath": dataset_path})
    executable_code = extract_code_block(response.content)

    #Execute the code safely using REPL tool
    print("\n Executing code via REPL...")
    repl_output = python_repl.invoke(executable_code)

    #parse REPL output to update state:
    requires_imputation = False
    if "0\n" not in repl_output and repl_output.strip() != "":
        requires_imputation = True

    return {
        "dataset_summary": repl_output,
        "missing_values_summary": repl_output,
        "requires_imputation": requires_imputation
    }

print("EDA Node defined.")

EDA Node defined.


In [9]:
# Create a dummy state to test the node
test_state: DataScienceState = {
    "dataset_path": "../data/iris.csv", # Replace with your actual local CSV path
    "dataset_summary": None,
    "missing_values_summary": None,
    "requires_imputation": False,
    "imputation_strategy": None,
    "scaling_strategy": None,
    "hyperparameters": None
}

# Run the node
updated_state = eda_node(test_state)

print("\n--- UPDATED STATE ---")
print("Requires Imputation:", updated_state["requires_imputation"])
print("\nDataset Summary Output:\n", updated_state["dataset_summary"])

--Entering EDA Node--


Python REPL can execute arbitrary code. Use with caution.



 Executing code via REPL...

--- UPDATED STATE ---
Requires Imputation: False

Dataset Summary Output:
 <class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB
None
sepal length (cm)    0
sepal width (cm)     0
petal length (cm)    0
petal width (cm)     0
dtype: int64



In [12]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

# Fetch as a pandas DataFrame
housing = fetch_california_housing(as_frame=True)

# Combine features and target into one DataFrame
df = housing.frame

# Save to your local folder
df.to_csv("../data/california_housing.csv", index=False)


In [13]:
# Create a dummy state to test the node
test_state2: DataScienceState = {
    "dataset_path": "../data/california_housing.csv", # Replace with your actual local CSV path
    "dataset_summary": None,
    "missing_values_summary": None,
    "requires_imputation": False,
    "imputation_strategy": None,
    "scaling_strategy": None,
    "hyperparameters": None
}

# Run the node
updated_state2 = eda_node(test_state2)

print("\n--- UPDATED STATE ---")
print("Requires Imputation:", updated_state2["requires_imputation"])
print("\nDataset Summary Output:\n", updated_state2["dataset_summary"])

--Entering EDA Node--

 Executing code via REPL...

--- UPDATED STATE ---
Requires Imputation: False

Dataset Summary Output:
 <class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   MedInc       20640 non-null  float64
 1   HouseAge     20640 non-null  float64
 2   AveRooms     20640 non-null  float64
 3   AveBedrms    20640 non-null  float64
 4   Population   20640 non-null  float64
 5   AveOccup     20640 non-null  float64
 6   Latitude     20640 non-null  float64
 7   Longitude    20640 non-null  float64
 8   MedHouseVal  20640 non-null  float64
dtypes: float64(9)
memory usage: 1.4 MB
None
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

